# 🔎 Georgia RAG — pipeline step by step

This notebook lets you walk through the whole pipeline by hand and inspect what comes out at each stage:

1. Check `.env` and settings
2. (optional) Fetch chat history
3. Raw messages
3b. Spam cleaning (quick-money / drugs / 18+)
4. Chunking (you can tweak parameters)
4b. Chunking research — sweep parameters & custom context
5. Indexing
6. Retrieval
7. RAG answer
8. Debug: which prompt is actually sent to GPT

In [1]:
# So that `import config` / `src.*` work from the notebooks/ folder, move to the project root
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

# Autoreload: edits in src/*.py are picked up without restarting the kernel
%load_ext autoreload
%autoreload 2

Working directory: /Users/aleksandra/Documents/GitHub/telegram-georgia-rag


## 1. Check `.env` and settings

Make sure the keys were picked up (values are masked).

In [2]:
import importlib, config
importlib.reload(config)

def mask(v):
    return (v[:4] + "…" + str(len(v)) + " chars") if v else "❌ not set"

print("OPENAI_API_KEY  :", mask(config.OPENAI_API_KEY))
print("BOT_TOKEN       :", mask(config.BOT_TOKEN))
print("TELEGRAM_API_ID :", config.TELEGRAM_API_ID or "❌ not set")
print("TELEGRAM_PHONE  :", config.TELEGRAM_PHONE or "❌ not set")
print()
print("Chats:", [c["username"] for c in config.CHATS])
print("Models:", config.EMBED_MODEL, "|", config.CHAT_MODEL)
print("Chunking: max gap =", config.CHUNK_MAX_GAP_MINUTES, "min, max size =", config.CHUNK_MAX_CHARS, "chars")

OPENAI_API_KEY  : sk-p…164 chars
BOT_TOKEN       : 8624…46 chars
TELEGRAM_API_ID : 38918746
TELEGRAM_PHONE  : +995XXXXXXXXX

Chats: ['helpgeorgia', 'ipgeorgiachat']
Models: text-embedding-3-small | gpt-4o-mini
Chunking: max gap = 10 min, max size = 1500 chars


## 2. (optional) Fetch chat history

If you already ran `uv run python -m src.ingest` in the terminal — skip this step.

The first run will ask for the confirmation code from Telegram (entered right in the notebook). After authorization a session file is created, so the code won't be needed again.

In [3]:
# Uncomment to fetch history directly from the notebook:
#
# from telethon import TelegramClient
# from src.ingest import ingest_chat
#
# client = TelegramClient("georgia_ingest", int(config.TELEGRAM_API_ID), config.TELEGRAM_API_HASH)
# await client.start(phone=config.TELEGRAM_PHONE or None)
# for chat in config.CHATS:
#     await ingest_chat(client, chat)
# await client.disconnect()

## 3. Raw messages

Look at what was fetched: how many messages and what they look like.

In [4]:
from src.preprocess import _load_raw

username = config.CHATS[0]["username"]
raw_path = config.RAW_DIR / f"{username}.jsonl"
print("File:", raw_path, "| exists:", raw_path.exists())

if raw_path.exists():
    msgs = _load_raw(raw_path)
    print("Total messages:", len(msgs))
    print("\nLast 5:")
    for m in msgs[-5:]:
        sender = m.get("sender") or "Anonymous"
        print(f"  [{m['date'][:16]}] {sender}: {m['text'][:90]}")
else:
    print("Fetch the history first (step 2 or `uv run python -m src.ingest`).")

File: /Users/aleksandra/Documents/GitHub/telegram-georgia-rag/data/raw/helpgeorgia.jsonl | exists: True
Total messages: 2602

Last 5:
  [2026-06-07T08:21] Anton: Добрый день! Кто ходил к леднику Гергети у подножия Казбека от Монастыря Гергети? Наскольк
  [2026-06-08T06:11] Vvk: Добрый день, подскажите контакты передержки для кошек( не дорогие), кошка домашняя
  [2026-06-08T06:26] Ксения: Доброго времени суток!
Кто-нибудь знает психиатров, которые выписывают рецепты в Тбилиси?
  [2026-06-08T15:40] Передержка котиков Тбилиси: написала вам в лс
  [2026-06-11T15:16] 𝕰𝖌𝖔𝖗: Если кто-то еще пользуется "фаэтоном" для поездок в Ереван - будьте бдительны пожалуйста ❤


## 3b. Spam cleaning

Before chunking we drop spam: "quick money" schemes, veiled drug ads from darkstores, and 18+ / escort posts. Patterns live in `src/spam.py` (`SPAM_PATTERNS`) — eyeball the removed examples below and tune them. This cell sets `msgs` to the cleaned list for everything that follows.

> In the pipeline this runs automatically (toggle with `config.FILTER_SPAM`).

In [5]:
from src.spam import filter_spam
from collections import Counter

clean, removed = filter_spam(msgs)
print(f"{len(msgs)} messages -> {len(clean)} clean, {len(removed)} spam removed")
print("by category:", dict(Counter(r["spam"] for r in removed)))

# A few examples of what got removed — eyeball these for false positives:
for r in removed[:8]:
    print(f"  [{r['spam']}] {r['text'][:100]}")

# Use the cleaned messages for everything below (chunking research, etc.):
msgs = clean

2602 messages -> 2537 clean, 65 spam removed
by category: {'money': 60, 'adult': 4, 'drugs': 1}
  [money] сдаю квартиру в Тбилиси срочно по хорошей цене , я собственник, пишите в лс
  [money] 👋🇬🇪🇬🇧🇷🇺
В авторский магазин сувениров и изделий ручной работы в центре Тбилиси требуется продавец-ко
  [money] 👋🇬🇪🇬🇧🇷🇺
В авторский магазин сувениров и изделий ручной работы в центре Тбилиси требуется продавец-ко
  [money] законно? ведь на иностранных номерах 50 в сутки, но не более 1000 при выезде
  [money] Батуми. Срочно ищем руководителя отдела продаж с хорошим опытом. Пишите в личку
  [money] Привет, я Няня! Меня зовут Натиа, 22 года, Тбилиси. 
Сижу с детьми около двух леь, плюс всегда сидел
  [money] Продаю айфон 14 про макс 256 гб в идеальнейшем состоянии. Черный цвет . Полный комплект и гарантия. 
  [money] Всем привет, вопрос, пишут что в Грузии на временный учет авто из России не любого года можно постав


## 4. Chunking — how messages are merged into dialogs

Build chunks and inspect the result. The `CHUNK_MAX_GAP_MINUTES` and `CHUNK_MAX_CHARS` parameters can be changed right here to compare.

In [6]:
from src.preprocess import chunk_messages

# Feel free to experiment with the parameters:
# config.CHUNK_MAX_GAP_MINUTES = 15
# config.CHUNK_MAX_CHARS = 2000

chunks = chunk_messages(msgs)
sizes = [len(c["text"]) for c in chunks]
print(f"{len(msgs)} messages -> {len(chunks)} chunks")
if sizes:
    print(f"Chunk size (chars): min {min(sizes)}, avg {sum(sizes)//len(sizes)}, max {max(sizes)}")

2537 messages -> 2091 chunks
Chunk size (chars): min 40, avg 431, max 3837


In [7]:
# Inspect 3 random chunks in full
import random
for c in random.sample(chunks, min(3, len(chunks))):
    print("=" * 70)
    print(c["link"], "| messages", c["first_msg_id"], "-", c["last_msg_id"])
    print(c["text"])

https://t.me/helpgeorgia/312136 | messages 312136 - 312136
Michael: Добрый день!

Подскажите пожалуйста сталкивался ли кто то с переездом из Тбилиси в Сербию с собакой ?

Хотел бы узнать, что из документов нужно 
И как транспортировать его, желательно наземным транспортом или компанией
https://t.me/helpgeorgia/309203 | messages 309203 - 309203
Print: ребята, кто знает откуда едут маршрутки тбилиси мцхета?
https://t.me/helpgeorgia/307280 | messages 307280 - 307280
Лера: Если в эмиграции ваше питание превратилось в череду срывов, диет и перееданий, а цифры на весах только растут, готова помочь вам выбраться из этого замкнутого круга.

Я Лера, нутрициолог, помогаю наладить отношения с едой и прийти к стройности. На моём канале вы найдёте:

• Рекомендации, как поддерживать сбалансированное питание
• Проверенные стратегии для постепенного обретения стройности без ограничений, запретов и диет.
• Подготовила для вас статью «Секреты стройности без подсчета калорий и диет»

Присоединяйтесь, кан

## 4b. Chunking research — sweep parameters & try custom context

All knobs are **arguments** to `chunk_messages`, so you can compare variants without editing `config` or restarting the kernel:

- `gap_minutes` — time window for grouping messages
- `max_chars` — chunk body size cap
- `context_max_chars` — cap on the quoted reply context
- `whole_burst` — pull the whole conversation around each ancestor (`True`) or just the single message (`False`)
- `chain_max` — how far up the reply chain to walk
- `context_fn` — full override to experiment with brand-new context logic

> Run the **"3. Raw messages"** cell above first so that `msgs` is loaded.

In [ ]:
from src.preprocess import chunk_messages, ancestor_context

def preview(msgs, n=3, max_show=900, **params):
    """Build chunks with the given params and show stats + a few chunks that
    actually carry reply context (the interesting ones)."""
    chunks = chunk_messages(msgs, **params)
    sizes = [len(c["text"]) for c in chunks]
    with_ctx = [c for c in chunks if "↪" in c["text"]]
    avg = sum(sizes) // len(sizes) if sizes else 0
    print(f"params: {params}")
    print(f"  {len(chunks)} chunks | avg {avg} chars | max {max(sizes) if sizes else 0} | with reply-context: {len(with_ctx)}")
    for c in with_ctx[:n]:
        print("-" * 70)
        print(c["link"])
        print(c["text"][:max_show])
    return chunks

# One configuration to start from:
_ = preview(msgs, n=2, gap_minutes=10, whole_burst=True, context_max_chars=2000)

In [ ]:
import itertools

print(f"{'gap':>4} {'ctx_max':>8} {'whole':>6} | {'chunks':>7} {'avg':>5} {'w/ctx':>6}")
for gap, ctx_max, whole in itertools.product([5, 10, 20], [800, 2000, 4000], [True, False]):
    chunks = chunk_messages(msgs, gap_minutes=gap, context_max_chars=ctx_max, whole_burst=whole)
    sizes = [len(c["text"]) for c in chunks]
    avg = sum(sizes) // len(sizes) if sizes else 0
    n_ctx = sum("↪" in c["text"] for c in chunks)
    print(f"{gap:>4} {ctx_max:>8} {str(whole):>6} | {len(chunks):>7} {avg:>5} {n_ctx:>6}")

### Experiment: custom `context_fn`

`context_fn(buf, by_id, burst_map) -> list[dict]` returns the messages to quote as context. Define your own right here (no autoreload needed — it lives in the notebook) and pass it via `context_fn=`. When you pass `context_fn`, the `whole_burst` / `context_max_chars` / `chain_max` args are ignored.

In [ ]:
def neighbours_around_ancestors(buf, by_id, burst_map, n=2, cap=2000):
    """For each reply parent outside the chunk, grab ±n messages around it by
    position (ignoring time). Pure notebook experiment — edit freely."""
    ids_sorted = sorted(by_id)
    pos = {mid: i for i, mid in enumerate(ids_sorted)}
    body = {m["msg_id"] for m in buf}
    seen, chars = set(), 0
    for m in buf:
        p = m.get("reply_to")
        if not p or p not in pos:
            continue
        i = pos[p]
        for j in range(max(0, i - n), min(len(ids_sorted), i + n + 1)):
            mid = ids_sorted[j]
            if mid in body or mid in seen:
                continue
            msg = by_id[mid]
            if chars + len(msg["text"]) > cap:
                continue
            seen.add(mid); chars += len(msg["text"])
    return [by_id[i] for i in sorted(seen)]

print("=== default (whole burst around ancestors) ===")
_ = preview(msgs, n=1)
print("\n=== custom: ±2 neighbours around ancestors (by position) ===")
_ = preview(msgs, n=1, context_fn=lambda b, i, m: neighbours_around_ancestors(b, i, m, n=2))

## 5. Indexing (embeddings → Chroma)

⚠️ This step spends OpenAI tokens (embeddings are cheap but not free). Run it once the chunks look good.

In [ ]:
# Indexing is usually more convenient to run as scripts (they read data/chunks/*.jsonl):
#   uv run python -m src.preprocess   # save chunks to disk
#   uv run python -m src.index        # build the index
#
# Or right here:
from src.index import main as build_index
from src.preprocess import main as build_chunks

build_chunks()   # data/chunks/*.jsonl
build_index()    # embeddings -> chroma_db/

In [ ]:
from src.store import get_collection
col = get_collection()
print("Records in collection:", col.count())

## 6. Retrieval

See which fragments are found for a question and how relevant they are (score closer to 1 = better).

In [ ]:
from src.retrieve import search

query = "как открыть ип в грузии"  # <- change the question (keep it in Russian)

for i, h in enumerate(search(query, k=5), 1):
    print(f"--- #{i}  score={h['score']:.3f}  {h['meta']['link']}")
    print(h["text"][:300])
    print()

## 7. RAG answer

The final answer from GPT based on the retrieved fragments + the list of sources.

In [ ]:
from src.rag import answer

res = answer("какие документы нужны для открытия ип")  # <- change the question

print(res["answer"])
print("\n——— Sources ———")
for s in res["sources"]:
    print(s["title"], "|", s["link"])

## 8. Debug: which prompt is actually sent to GPT

Useful to understand why the model answered the way it did, and to tweak the system prompt in `src/rag.py`.

In [ ]:
from src.rag import _build_context, SYSTEM_PROMPT
from src.retrieve import search

q = "как получить внж"
hits = search(q, k=4)

print("### SYSTEM PROMPT ###\n")
print(SYSTEM_PROMPT)
print("\n### CONTEXT (fragments) ###\n")
print(_build_context(hits))